In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import os
import glob

print("All libraries imported successfully!")

In [ ]:
files = glob.glob('/kaggle/input/**/*.csv', recursive=True)

print("Available CSV files:")

for file in files:
    print(file)

In [ ]:
data_path = files[0]
df = pd.read_csv(data_path)

print("Dataset loaded successfully!")
print("Dataset Shape:", df.shape)

print("\nFirst 5 Rows:")
display(df.head())

print("\nDataset Information:")
df.info()

print("\nMissing Values:")
print(df.isnull().sum())

In [ ]:
# Separate features and target

X = df.drop(columns=['Concrete compressive strength ']).values

y = df['Concrete compressive strength '].values.reshape(-1, 1)

print("Features shape:", X.shape)
print("Target shape:", y.shape)


# Split data into training and testing sets

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


# Scale input features

scaler_X = StandardScaler()

X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled = scaler_X.transform(X_test)


# Scale target variable

scaler_y = StandardScaler()

y_train_scaled = scaler_y.fit_transform(y_train)
y_test_scaled = scaler_y.transform(y_test)


# Generate polynomial features

poly = PolynomialFeatures(degree=2, include_bias=False)

X_train_poly = poly.fit_transform(X_train_scaled)
X_test_poly = poly.transform(X_test_scaled)


print("\nPolynomial Regression Results:")
print("Original features:", X_train_scaled.shape[1])
print("Polynomial features:", X_train_poly.shape[1])
print("Training data shape:", X_train_poly.shape)
print("Testing data shape:", X_test_poly.shape)

In [ ]:
# Prediction function

def predict(X, weights, bias):
    return X @ weights + bias


# Loss function

def compute_loss(y_true, y_pred):
    return np.mean((y_pred - y_true) ** 2)


# Gradient calculation

def compute_gradients(X, y, weights, bias):

    m = X.shape[0]

    predictions = predict(X, weights, bias)

    errors = predictions - y

    dw = (2 / m) * (X.T @ errors)

    db = (2 / m) * np.sum(errors)

    return dw, db


print("Model functions defined successfully!")

In [ ]:

def train_sgd(X, y, learning_rate=0.001, momentum=0.9, epochs=100):
    m, n = X.shape

    weights = np.zeros((n, 1))
    bias = 0.0

    # Initialize momentum velocities
    velocity_w = np.zeros_like(weights)
    velocity_b = 0.0

    loss_history = []

    for epoch in range(epochs):

        # Shuffle training samples
        indices = np.random.permutation(m)

        X_shuffled = X[indices]
        y_shuffled = y[indices]

        # Update parameters using one sample at a time
        for i in range(m):

            xi = X_shuffled[i:i+1]
            yi = y_shuffled[i:i+1]

            dw, db = compute_gradients(
                xi, yi, weights, bias
            )

            # Apply Momentum
            velocity_w = momentum * velocity_w + dw
            velocity_b = momentum * velocity_b + db

            # Update parameters
            weights -= learning_rate * velocity_w
            bias -= learning_rate * velocity_b

        # Calculate loss after each epoch
        predictions = predict(X, weights, bias)
        loss = compute_loss(y, predictions)

        loss_history.append(loss)

    return weights, bias, loss_history


# Train SGD with Momentum
np.random.seed(42)

sgd_weights, sgd_bias, sgd_loss = train_sgd(
    X_train_poly,
    y_train_scaled,
    learning_rate=0.0001,
momentum=0.9,
epochs=100
)

print("Stochastic Gradient Descent (SGD) with Momentum")
print("Momentum:", 0.9)
print("Final Training Loss:", sgd_loss[-1])


In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(sgd_loss, label='SGD', color='orange')

plt.title('Training Loss - Stochastic Gradient Descent')
plt.xlabel('Epochs')
plt.ylabel('Mean Squared Error (MSE)')

plt.legend()
plt.grid(True)
plt.show()

In [ ]:

predictions_scaled = predict(
    X_test_poly, sgd_weights, sgd_bias
)

predictions = scaler_y.inverse_transform(predictions_scaled)
actual = scaler_y.inverse_transform(y_test_scaled)

mse = mean_squared_error(actual, predictions)
rmse = np.sqrt(mse)
mae = mean_absolute_error(actual, predictions)
r2 = r2_score(actual, predictions)

results = pd.DataFrame([{
    "Optimizer": "SGD with Momentum",
    "MSE": mse,
    "RMSE": rmse,
    "MAE": mae,
    "R2 Score": r2
}])

display(results.round(4))


In [ ]:
plt.figure(figsize=(8, 6))

plt.scatter(actual, predictions, alpha=0.6, label='Predictions')

# Ideal prediction line
min_value = min(actual.min(), predictions.min())
max_value = max(actual.max(), predictions.max())

plt.plot(
    [min_value, max_value],
    [min_value, max_value],
    'r--',
    label='Perfect Prediction'
)

plt.title('Actual vs Predicted - SGD')
plt.xlabel('Actual Concrete Strength')
plt.ylabel('Predicted Concrete Strength')

plt.legend()
plt.grid(True)
plt.show()


### Conclusion

The SGD optimizer was implemented with Momentum to improve the parameter update process by incorporating information from previous gradients.

The model was trained using a learning rate of 0.0001, a momentum coefficient of 0.9, and 100 epochs.

The final training loss was 0.2002.

The evaluation results were:
- MSE: 58.2073
- RMSE: 7.6294
- MAE: 6.0610
- R2 Score: 0.7741

The Actual vs Predicted plot shows that most predictions follow the general trend of the actual concrete strength values.

The results indicate that SGD with Momentum achieved reasonable predictive performance, although some prediction errors and fluctuations in training loss remain.
